In [153]:
from nnfs.datasets import spiral_data
import numpy as np
import nnfs
nnfs.init()
import matplotlib.pyplot as plt

In [154]:
X, y = spiral_data(samples=500,classes=3)
X_train = X.T

In [155]:
class Layer_dense:
    def __init__(self,inputs,neurons):
        self.weights = np.random.randn(neurons,inputs) * np.sqrt(2 / inputs)
        self.biases = np.zeros((neurons,1))
    def forward(self,x):
        self.inputs = x
        self.output = self.weights @ x + self.biases
        return self.output

class ReluActivation:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = np.maximum(0,inputs)
        return self.output
    
class SoftmaxActivation:
    def forward(self,inputs):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs,axis=0,keepdims=True))
        probabilities = exp_values/np.sum(exp_values,axis=0,keepdims=True)
        self.output = probabilities
        return self.output

class Loss:
    def forward(self,y_pred,y_true):
        samples = y_pred.shape[1]
        if y_true.ndim == 2:
            y_true = np.argmax(y_true, axis=0)
        correct_confidence = y_pred[y_true,np.arange(samples)]
        correct_confidence = np.clip(
            correct_confidence,
            1e-7,
            1 - 1e-7
        )
        loss = -np.log(correct_confidence)
        return np.mean(loss)

In [156]:
# training data contains two features which is the x , y co ordinates of a point
layer_1 = Layer_dense(2,64)
layer_2 = Layer_dense(64,64)
layer_3 = Layer_dense(64,3)
relu1 = ReluActivation()
relu2 =ReluActivation()
softmax = SoftmaxActivation()
loss_function = Loss()

In [157]:
print("X_train:", X_train.shape)
print("layer_1.weights:", layer_1.weights.shape)
print("layer_1.bias:", layer_1.bias.shape if hasattr(layer_1, "bias") else layer_1.biases.shape)

X_train: (2, 1500)
layer_1.weights: (64, 2)
layer_1.bias: (64, 1)


In [158]:
np.random.seed(42)

In [159]:
learning_rate = 0.1
epochs = 5000

accuracy_dict = {
    "accuracy": [],
    "w1": [],
    "w2": [],
    "w3": [],
    "b1": [],
    "b2": [],
    "b3": []
}

best_accuracy = 0
best_params = {}

for epoch in range(epochs):
    #initialization
    inputs = X_train

    output1 = relu1.forward(layer_1.forward(inputs))
    output2 = relu2.forward(layer_2.forward(output1))
    output3 = softmax.forward(layer_3.forward(output2))

    predictions = output3

    loss = loss_function.forward(predictions,y)

    predicted_class = np.argmax(predictions,axis=0)
    accuracy = np.mean(predicted_class==y)

    #backpropagation 
    m = X_train.shape[1]

    dz3 = predictions.copy()
    dz3[y, np.arange(m)] -= 1
    dw3 = (1/m) * (dz3 @ output2.T)
    db3 = (1/m) * np.sum(dz3,axis=1,keepdims=True)
    da2 = layer_3.weights.T @ dz3
    
    dz2 = da2.copy()
    dz2[relu2.inputs <= 0] = 0
    dw2 = (1/m) * (dz2 @ output1.T)
    db2 = (1/m) * np.sum(dz2,axis=1,keepdims=True)
    da1 = layer_2.weights.T @ dz2

    dz1 = da1.copy()
    dz1[relu1.inputs <= 0] = 0
    dw1 = (1/m) * (dz1 @ inputs.T)
    db1 = (1/m) * np.sum(dz1,axis=1,keepdims=True)

    #updating weights and biases
    layer_1.weights -= learning_rate * dw1
    layer_1.biases -= learning_rate * db1

    layer_2.weights -= learning_rate * dw2
    layer_2.biases -= learning_rate * db2

    layer_3.weights -= learning_rate * dw3
    layer_3.biases -= learning_rate *db3

    if epoch % 500 == 0:
        print(
            f"Epoch: {epoch}, "
            f"Loss: {loss:.4f}, "
            f"Accuracy: {accuracy:.4f}"
        )
    accuracy_dict["accuracy"].append(accuracy)

    accuracy_dict["w1"].append(layer_1.weights.copy())
    accuracy_dict["w2"].append(layer_2.weights.copy())
    accuracy_dict["w3"].append(layer_3.weights.copy())

    accuracy_dict["b1"].append(layer_1.biases.copy())
    accuracy_dict["b2"].append(layer_2.biases.copy())
    accuracy_dict["b3"].append(layer_3.biases.copy())

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = {
            "w1": layer_1.weights.copy(), "b1": layer_1.biases.copy(),
            "w2": layer_2.weights.copy(), "b2": layer_2.biases.copy(),
            "w3": layer_3.weights.copy(), "b3": layer_3.biases.copy(),
        }



Epoch: 0, Loss: 1.1412, Accuracy: 0.3253
Epoch: 500, Loss: 1.0410, Accuracy: 0.4353
Epoch: 1000, Loss: 0.9701, Accuracy: 0.4853
Epoch: 1500, Loss: 0.7819, Accuracy: 0.7040
Epoch: 2000, Loss: 0.5641, Accuracy: 0.8020
Epoch: 2500, Loss: 0.4640, Accuracy: 0.8187
Epoch: 3000, Loss: 0.3393, Accuracy: 0.8860
Epoch: 3500, Loss: 0.3145, Accuracy: 0.8840
Epoch: 4000, Loss: 0.2902, Accuracy: 0.8893
Epoch: 4500, Loss: 0.2713, Accuracy: 0.8967


In [160]:
print(f"Best accuracy: {best_accuracy:.4f}")

Best accuracy: 0.9013


In [161]:
X_test, y_test = spiral_data(samples=100, classes=3)
X_test = X_test.T

layer_1.weights = best_params["w1"];  
layer_1.biases = best_params["b1"]
layer_2.weights = best_params["w2"];  
layer_2.biases = best_params["b2"]
layer_3.weights = best_params["w3"]; 
layer_3.biases = best_params["b3"]

o1 = relu1.forward(layer_1.forward(X_test))
o2 = relu2.forward(layer_2.forward(o1))
o3 = softmax.forward(layer_3.forward(o2))


test_loss     = loss_function.forward(o3, y_test)
predicted     = np.argmax(o3, axis=0)
test_accuracy = np.mean(predicted == y_test)

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss:     0.2893
Test Accuracy: 0.8767


In [163]:
sample_idx = 10

sample    = X_test[:, sample_idx].reshape(2, 1)   
true_label = y_test[sample_idx]

o1 = relu1.forward(layer_1.forward(sample))
o2 = relu2.forward(layer_2.forward(o1))
o3 = softmax.forward(layer_3.forward(o2))

class_names = {0: "Red", 1: "Blue", 2: "Green"}
predicted_label = np.argmax(o3, axis=0)[0]

print(f"Predicted : {class_names[predicted_label]}")
print(f"True Label: {class_names[true_label]}")
print(f"\nClass Probabilities:")
for i, name in class_names.items():
    print(f"  {name:6s}: {o3[i, 0]*100:.2f}%")

Predicted : Red
True Label: Red

Class Probabilities:
  Red   : 98.87%
  Blue  : 0.06%
  Green : 1.07%
